# Aphasia classifier — quick start

This notebook walks through the whole workflow:

1. look at the data and the features,
2. train the classifier,
3. read the results,
4. score new transcripts,
5. inspect the features of any sentence you type.

Run the cells top to bottom (Shift+Enter). Cells that start with `!` run a script exactly as you would from the terminal.

Two tasks are included — `cinderella` (the default) and `sandwich`. To use the sandwich corpus instead, set `TASK` in the first code cell below.

**Before you start:** install the package as described in `README.md`, and open this notebook from inside the `aphasia_classifier/` folder (the first cell checks).

In [ ]:
# Which task to work on: "cinderella" or "sandwich" (or a folder you added under data/).
TASK = "cinderella"

# Make the package importable and move to its folder, wherever the notebook was opened from.
import os, sys
from pathlib import Path
os.environ["APHASIA_TASK"] = TASK       # every script below picks this up

here = Path.cwd()
package = here if (here / "src" / "config.py").exists() else here.parent
os.chdir(package)
sys.path.insert(0, str(package / "src"))

import config
print("working in:", Path.cwd(), "| task:", config.TASK)
print("transcripts:", config.TRANSCRIPTS_FILE.name, "| features:", config.FEATURES_FILE.name)
print("the classifier will use", len(config.FEATURES), "features:")
print(config.FEATURES)

## 1. The data

`data/<task>/transcripts.json` holds one narrative per participant with a diagnostic group. `data/<task>/features.csv` holds the numeric features for each of them (already computed — see step 5 to recompute).

**The transcripts are shared separately.** They are AphasiaBank data and are not in the git repository. If the next cell stops with `FileNotFoundError`, unpack `aphasia_classifier_transcripts.zip` in the `aphasia_classifier/` folder (README, *Installation §5*) and run it again.

In [ ]:
import json
import pandas as pd

transcripts = json.loads(config.TRANSCRIPTS_FILE.read_text())
print(len(transcripts), "transcripts")
print(pd.Series([t[config.LABEL_COLUMN] for t in transcripts]).value_counts(), "\n")

example = transcripts[0]
print(example[config.ID_COLUMN], "|", example[config.LABEL_COLUMN])
print(example[config.TEXT_COLUMN][:400], "...")

In [ ]:
features = pd.read_csv(config.FEATURES_FILE)
print(features.shape[0], "rows x", features.shape[1], "columns")

# The 16 features the classifier uses, averaged per group. Notice how they
# already separate the groups before any training happens.
features.groupby(config.LABEL_COLUMN)[config.FEATURES].mean().round(2).T

### Reading the transcripts by group — where a new feature starts

The transcript text is the only thing the classifier has for a *new* speaker, so any new feature has to be computable from it. The cells below show the loop: read narratives from each group side by side, turn an observation into a number, and check whether that number separates the groups. `data/README.md` describes the text conventions (one utterance per sentence, commas as separate tokens, repetitions kept as spoken).

In [ ]:
df = pd.DataFrame(transcripts)

# two narratives from each group, side by side, to look for differences
for group, rows in df.groupby(config.LABEL_COLUMN):
    print(f"===== {group} =====")
    for text in rows[config.TEXT_COLUMN].head(2):
        print(" ", text[:300], "...\n")

In [ ]:
# A candidate feature in a few lines: what share of the utterances are very
# short (three words or fewer)? Telegraphic speech should push this up.
def short_utterance_rate(text):
    utterances = [u.split() for u in text.split(". ") if u.strip()]
    return sum(len(u) <= 3 for u in utterances) / len(utterances)

df["short_utterance_rate"] = df[config.TEXT_COLUMN].map(short_utterance_rate)

# If the groups do not differ here, the feature will not help the classifier.
df.groupby(config.LABEL_COLUMN)["short_utterance_rate"].describe().round(3)

## 2. Train

Each *seed* trains one model on a different random split; the final answer averages them. Two seeds is enough to see it work (about 15 minutes on a GPU). The paper used 20 — change `N_SEEDS` in `src/config.py` or pass `--n-seeds 20` here.

To change **which features** are used, edit the `FEATURES` list in `src/config.py` before running this cell.

In [ ]:
!python src/train.py --n-seeds 2

## 3. Results

`models/results.json` has the metrics; `models/ensemble_predictions.csv` has the classifier's probability for every transcript, which lets you look at the mistakes.

In [ ]:
results = json.loads((config.MODELS_DIR / "results.json").read_text())
print("ensemble:", {k: round(v, 3) if isinstance(v, float) else v for k, v in results["ensemble"].items()})

predictions = pd.read_csv(config.MODELS_DIR / "ensemble_predictions.csv")
print("\nmean P(aphasia) by group:")
print(predictions.groupby(config.LABEL_COLUMN)["P_aphasia"].mean().round(3))

print("\nthe controls the classifier was most unsure about:")
predictions[predictions[config.LABEL_COLUMN] == config.CONTROL_LABEL].nlargest(5, "P_aphasia")

## 4. Score new transcripts

`predict.py` recomputes all the features for a new text and runs every trained model. It needs the surprisal language model (Llama-2 by default — see README, Installation §4 — or set `SURPRISAL_MODEL = "gpt2-xl"` in `config.py`).

In [ ]:
!python src/predict.py --csv data/{TASK}/example_new_transcripts.csv --text-column transcript

pd.read_csv(f"data/{TASK}/example_new_transcripts_scored.csv")[["name", "P_aphasia", "predicted_class"]]

In [ ]:
# Or type a transcript directly:
!python src/predict.py --text "Cinderella. um. the girl. she uh she clean. stepmother mean. ball. shoe lost. prince find."

## 5. Look at the features of any sentence

The two cheapest feature groups need no models at all, so you can inspect them interactively. This is also the place to start if you want to **design your own feature**: write a function like these in a new file under `src/features/`.

In [ ]:
from features import narrative, repetition

text = "I can you I can you I can you. the story. I can you. the story of the story. I can you."

print("narrative :", narrative.compute(text))
profile = repetition.repetition_profile(text)
print("repetition:", {k: profile[k] for k in repetition.COLUMNS})
print("   shortest repeated unit:", repr(profile["rep_shortest_str"]))

### Rebuilding the feature table

If you change a feature definition, add your own group, or bring your own transcripts, rebuild `features.csv`. Rebuilding a subset of groups keeps every other column (including ones you added in Excel).

In [ ]:
# The cheap groups run in seconds. Add similarity (~1 min) or surprisal (GPU + language model) when you need them.
!python src/build_features.py --groups narrative,repetition

## Where to go next

* **Different features:** edit `FEATURES` in `src/config.py`, then re-run step 2. Every column of `data/features.csv` is available, including CLAN's gold measures (`gold_*`).
* **Different models:** `TEXT_ENCODER`, `SURPRISAL_MODEL`, `SIMILARITY_MODEL` in `src/config.py`.
* **Your own feature:** *Designing a new feature from the transcripts* in `data/README.md`, then step 5 above.
* **Your own data:** see *Use your own transcripts* in `README.md`.
* **Report-quality numbers:** `!python src/train.py --n-seeds 20`.